# 색상모델

In [ ]:
# 필요한 라이브러리들
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score, hamming_loss, roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os
import pickle
from google.colab import drive

drive.mount('/content/drive')

# 설정값들
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
EPOCHS = 60
SEED = 42
color_classes = ['Beige', 'Black', 'Blue', 'Brown', 'Gray', 'Green', 'Orange', 'Pink', 'Purple', 'Red', 'White', 'Yellow']
model_save_path = "/content/drive/MyDrive/윤서/color_classifier.keras"
data_save_dir = "/content/drive/MyDrive/윤서/윤서"

print(f"⚠️ 입력 이미지 크기: {IMG_SIZE}")
print(f"⚠️ 채널: RGB (3채널)")

# =========================
# 1) 저장된 Numpy 데이터 로드
# =========================
print("🔄 저장된 Numpy 배열(.npy) 로드 중...")

images_path = os.path.join(data_save_dir, "color_images_array.npy")
labels_path = os.path.join(data_save_dir, "color_labels_array.npy")

try:
    images = np.load(images_path)
    labels = np.load(labels_path)
    print(f"✅ Numpy 배열 로드 완료:")
    print(f"   - Images: {images_path} ({images.shape})")
    print(f"   - Labels: {labels_path} ({labels.shape})")
except FileNotFoundError:
    print(f"🚨 에러: Numpy 파일을 찾을 수 없습니다.")
    raise
except Exception as e:
    print(f"🚨 에러: Numpy 파일 로드 중 오류 발생: {e}")
    raise

# 데이터 타입 확인 및 변환
if images.dtype != np.float32:
    print(f"⚠️ 이미지 데이터 타입을 {images.dtype}에서 float32로 변환 중...")
    images = images.astype(np.float32)

if labels.dtype != np.int32:
    print(f"⚠️ 라벨 데이터 타입을 {labels.dtype}에서 int32로 변환 중...")
    labels = labels.astype(np.int32)

# 이미지 값 범위 확인
print(f"이미지 값 범위: [{images.min():.2f}, {images.max():.2f}]")

# 값이 0-1 범위가 아니라면 정규화
if images.max() > 1.0:
    print("⚠️ 이미지가 0-1 범위를 벗어났습니다. 0-1 범위로 정규화 중...")
    images = images / 255.0
elif images.min() < 0.0:
    print(f"⚠️ 이미지 최소값이 0보다 작습니다. ({images.min()}).")

print(f"\n📊 데이터 정보:")
print(f"   - 총 샘플 수: {len(images)}")
print(f"   - 이미지 크기: {images.shape[1:]}")
print(f"   - 클래스 수: {len(color_classes)}")

# 라벨 분포 확인
print(f"\n📈 클래스별 라벨 분포:")
label_counts = labels.sum(axis=0)
for i, color in enumerate(color_classes):
    print(f"   {color}: {label_counts[i]}")

# =========================
# 2) 데이터 분할
# =========================
print(f"\n🔄 데이터 전처리 및 분할...")

stratify_labels = labels.sum(axis=1)
stratify_labels = np.clip(stratify_labels, 0, 5)

try:
    X_temp, X_test, y_temp, y_test = train_test_split(
        images, labels, test_size=0.2, random_state=SEED,
        stratify=stratify_labels
    )

    stratify_temp = y_temp.sum(axis=1)
    stratify_temp = np.clip(stratify_temp, 0, 5)

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.1, random_state=SEED,
        stratify=stratify_temp
    )
except:
    print("⚠️ 층화 추출 실패, 일반 분할 사용")
    X_temp, X_test, y_temp, y_test = train_test_split(
        images, labels, test_size=0.2, random_state=SEED
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.1, random_state=SEED
    )

del images, labels
import gc
gc.collect()
print("메모리 최적화: 원본 'images', 'labels' 배열 삭제 완료.")

print(f"데이터 분할 완료:")
print(f"- Train: {X_train.shape}")
print(f"- Validation: {X_val.shape}")
print(f"- Test: {X_test.shape}")

# 각 세트별 라벨 분포
print(f"\n각 세트별 라벨 분포:")
for name, y_set in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    label_dist = y_set.sum(axis=0)
    print(f"{name}: {dict(zip(color_classes, label_dist))}")

# EfficientNet 전처리
print(f"\n🔧 EfficientNet 전처리 적용...")
X_train = preprocess_input(X_train * 255.0)
X_val = preprocess_input(X_val * 255.0)
X_test = preprocess_input(X_test * 255.0)

print(f"전처리 후 값 범위:")
print(f"   - Train: [{X_train.min():.2f}, {X_train.max():.2f}]")

# =========================
# 3) 데이터 증강
# =========================
print(f"\n🎨 EfficientNet 최적화 데이터 증강...")
train_datagen = ImageDataGenerator(
    rotation_range=25,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator()

train_gen = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
val_gen = val_datagen.flow(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)

# =========================
# 4) EfficientNetB3 모델 정의 - ⚠️ 수정된 버전
# =========================
print(f"\n🏗️ EfficientNet-B3 모델 구성...")

# ⚠️ 수정: Input 레이어를 먼저 생성
inputs = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name='input_layer')

# ⚠️ 수정: input_tensor로 전달
base_model = EfficientNetB3(
    weights='imagenet',
    include_top=False,
    input_tensor=inputs  # ⚠️ 핵심: input_tensor 사용!
)
base_model.trainable = False

# Functional API로 모델 구성
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.6)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(color_classes), activation='sigmoid')(x)

model = models.Model(inputs=inputs, outputs=outputs, name='EfficientNet_ColorClassifier')

print("EfficientNetB3 모델 구조:")
model.summary()
print(f"📊 총 파라미터: {model.count_params():,}")
print(f"\n⚠️ 확인: 입력 shape = {model.input_shape}")
print(f"⚠️ 확인: 출력 shape = {model.output_shape}")

# =========================
# 5) 1단계 학습 (Frozen Backbone) - 30 에포크
# =========================
print("\n🚀 1단계 학습 시작: Frozen Backbone (30 에포크)")

base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    ),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

stage1_callbacks = [
    EarlyStopping(
        patience=12,
        restore_best_weights=True,
        monitor='val_loss',
        verbose=1
    ),
    ReduceLROnPlateau(
        patience=7,
        factor=0.3,
        monitor='val_loss',
        min_lr=1e-8,
        verbose=1
    ),
    ModelCheckpoint(
        model_save_path.replace('.keras', '_stage1.keras'),
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
]

stage1_epochs = 30
print(f"1단계 학습 에포크: {stage1_epochs}")

history_stage1 = model.fit(
    train_gen,
    epochs=stage1_epochs,
    validation_data=val_gen,
    callbacks=stage1_callbacks,
    verbose=1
)

print("✅ 1단계 학습 완료!")

# =========================
# 6) 2단계 학습 (Fine-tuning) - 30 에포크
# =========================
print("\n🚀 2단계 학습 시작: Fine-tuning (30 에포크)")

base_model.trainable = True
total_layers = len(base_model.layers)
freeze_until = total_layers // 2

for i, layer in enumerate(base_model.layers):
    layer.trainable = i >= freeze_until

print(f"Fine-tuning: {total_layers - freeze_until}/{total_layers} 층 해제")

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00005,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    ),
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

trainable_count = sum([layer.trainable for layer in model.layers])
print(f"Fine-tuning할 레이어 수: {trainable_count}")

stage2_callbacks = [
    EarlyStopping(
        patience=12,
        restore_best_weights=True,
        monitor='val_loss',
        verbose=1
    ),
    ReduceLROnPlateau(
        patience=7,
        factor=0.3,
        monitor='val_loss',
        min_lr=1e-8,
        verbose=1
    ),
    ModelCheckpoint(
        model_save_path,
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
]

stage2_epochs = 30
print(f"2단계 학습 에포크: {stage2_epochs}")
print(f"시작 에포크: {stage1_epochs}")

print("\n2단계 학습 시작 전 확인:")
print(f"- Base model trainable: {base_model.trainable}")
print(f"- Trainable layers in base model: {sum(1 for layer in base_model.layers if layer.trainable)}")

history_stage2 = model.fit(
    train_gen,
    epochs=stage1_epochs + stage2_epochs,
    validation_data=val_gen,
    callbacks=stage2_callbacks,
    verbose=1,
    initial_epoch=stage1_epochs
)

print("✅ 2단계 학습 완료!")

# =========================
# 7) 평가
# =========================
print("\n📊 모델 평가 중...")
y_prob = model.predict(X_test, batch_size=BATCH_SIZE)
y_pred = (y_prob >= 0.5).astype(int)

print("\n===== Classification Report (threshold=0.5) =====")
print(classification_report(y_test, y_pred, target_names=color_classes, zero_division=0))

prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(
    y_test, y_pred, average='micro', zero_division=0)
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average='macro', zero_division=0)
prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted', zero_division=0)

subset_acc = accuracy_score(y_test, y_pred)
hamm = hamming_loss(y_test, y_pred)

print("\n===== Aggregated Metrics =====")
print(f"Subset Accuracy (Exact Match): {subset_acc:.4f}")
print(f"Hamming Loss: {hamm:.4f}")
print(f"Micro  - Precision: {prec_micro:.4f} | Recall: {rec_micro:.4f} | F1: {f1_micro:.4f}")
print(f"Macro  - Precision: {prec_macro:.4f} | Recall: {rec_macro:.4f} | F1: {f1_macro:.4f}")
print(f"Weighted - Precision: {prec_weighted:.4f} | Recall: {rec_weighted:.4f} | F1: {f1_weighted:.4f}")

try:
    auc_macro = roc_auc_score(y_test, y_prob, average='macro')
    auc_micro = roc_auc_score(y_test, y_prob, average='micro')
    print(f"ROC-AUC (Macro): {auc_macro:.4f}")
    print(f"ROC-AUC (Micro): {auc_micro:.4f}")
except Exception as e:
    print(f"ROC-AUC 계산 스킵: {e}")

# =========================
# 8) 모델 저장
# =========================
model.save(model_save_path)
print(f"📁 최종 모델 저장: {model_save_path}")
print(f"⚠️ 입력 shape 확인: {model.input_shape}")
print(f"⚠️ 출력 shape 확인: {model.output_shape}")

history_path = os.path.join(data_save_dir, "training_history.pkl")
try:
    with open(history_path, 'wb') as f:
        pickle.dump({'stage1': history_stage1.history, 'stage2': history_stage2.history}, f)
    print(f"📊 학습 히스토리 저장: {history_path}")
except Exception as e:
    print(f"🚨 학습 히스토리 저장 실패: {e}")

print(f"\n✅ 학습 완료!")
print(f"📈 1단계 모델: {model_save_path.replace('.keras', '_stage1.keras')}")
print(f"📈 최종 모델: {model_save_path}")
print(f"📋 학습 방식: EfficientNetB3 2단계 학습")
print(f"   - 1단계 (Frozen): 30 에포크")
print(f"   - 2단계 (Fine-tuning): 30 에포크")
print(f"   - 총 에포크: 60")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠️ 입력 이미지 크기: (256, 256)
⚠️ 채널: RGB (3채널)
🔄 저장된 Numpy 배열(.npy) 로드 중...
✅ Numpy 배열 로드 완료:
   - Images: /content/drive/MyDrive/윤서/윤서/color_images_array.npy ((20897, 256, 256, 3))
   - Labels: /content/drive/MyDrive/윤서/윤서/color_labels_array.npy ((20897, 12))
이미지 값 범위: [0.00, 1.00]

📊 데이터 정보:
   - 총 샘플 수: 20897
   - 이미지 크기: (256, 256, 3)
   - 클래스 수: 12

📈 클래스별 라벨 분포:
   Beige: 2013
   Black: 6496
   Blue: 4759
   Brown: 1388
   Gray: 3031
   Green: 1727
   Orange: 407
   Pink: 1127
   Purple: 494
   Red: 1268
   White: 5219
   Yellow: 740

🔄 데이터 전처리 및 분할...
메모리 최적화: 원본 'images', 'labels' 배열 삭제 완료.
데이터 분할 완료:
- Train: (15045, 256, 256, 3)
- Validation: (1672, 256, 256, 3)
- Test: (4180, 256, 256, 3)

각 세트별 라벨 분포:
Train: {'Beige': np.int64(1455), 'Black': np.int64(4670), 'Blue': np.int64(3442), 'Brown': np.int64(994), 'Gray': np.int64(2174), 'Green': np.int64(1233)

Model: "EfficientNet_ColorClassifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 256, 256,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 256, 256,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 256, 256,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 257, 257,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 128, 128,  │      1,080 │ stem_conv_pad[0]… │
│                     │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 128, 128,  │        160 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 128, 128,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 128, 128,  │        360 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 128, 128,  │        160 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 128, 128,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 40)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 40)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 10)  │        410 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 40)  │        440 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 128, 128,  │          0 │ block1a_activati… │
│ (Multiply)          │ 40)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 128, 128,  │        960 │ block1a_se_excit

 Total params: 11,707,963 (44.66 MB)

 Trainable params: 922,892 (3.52 MB)

 Non-trainable params: 10,785,071 (41.14 MB)

📊 총 파라미터: 11,707,963

⚠️ 확인: 입력 shape = (None, 256, 256, 3)
⚠️ 확인: 출력 shape = (None, 12)

🚀 1단계 학습 시작: Frozen Backbone (30 에포크)
1단계 학습 에포크: 30


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 540ms/step - accuracy: 0.1933 - loss: 0.6013 - precision: 0.1723 - recall: 0.3788
Epoch 1: val_loss improved from inf to 0.27465, saving model to /content/drive/MyDrive/윤서/color_classifier_stage1.keras
471/471 ━━━━━━━━━━━━━━━━━━━━ 360s 630ms/step - accuracy: 0.1935 - loss: 0.6009 - precision: 0.1724 - recall: 0.3785 - val_accuracy: 0.3630 - val_loss: 0.2746 - val_precision: 0.7597 - val_recall: 0.1364 - learning_rate: 0.0010
Epoch 2/30
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.3181 - loss: 0.3030 - precision: 0.5105 - recall: 0.1771
Epoch 2: val_loss improved from 0.27465 to 0.26340, saving model to /content/drive/MyDrive/윤서/color_classifier_stage1.keras
471/471 ━━━━━━━━━━━━━━━━━━━━ 222s 471ms/step - accuracy: 0.3181 - loss: 0.3030 - precision: 0.5106 - recall: 0.1771 - val_accuracy: 0.3840 - val_loss: 0.2634 - val_precision: 0.7015 - val_recall: 0.1783 - learning_rate: 0.0010
Epoch 3/30
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/ste

#분류모델 데이터 넘파이 배열 변환

In [ ]:
#분류모델 데이터 넘파이 배열 변환

In [ ]:
# -*- coding: utf-8 -*-
"""
데이터 로딩 및 Numpy 저장 스크립트
학습 전 데이터를 전처리하여 .npy 파일로 저장합니다.
"""

# ✅ 1. Google Drive 마운트
print("Setting up environment...")
from google.colab import drive
import os
drive.mount('/content/drive')
print("✅ Environment setup complete.")

# ✅ 2. 필수 라이브러리 임포트
print("\nImporting libraries...")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from PIL import Image
from tqdm import tqdm
import pandas as pd
import gc

from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

Image.MAX_IMAGE_PIXELS = None
print("✅ Libraries imported successfully.")

# ✅ 3. 경로 및 파라미터 설정
print("\nConfiguring paths and parameters...")
original_data_path = "/content/drive/MyDrive/윤서/윤서/dataset3/images"
new_data_path = "/content/drive/MyDrive/윤서/윤서/retraining_data"
output_dir = "/content/drive/MyDrive/윤서/윤서/preprocessed_data"  # numpy 파일 저장 경로
os.makedirs(output_dir, exist_ok=True)

class_names = [
    'blazer', 'cardigan', 'coat', 'cotton pants', 'denim', 'dress', 'hood',
    'longpedding', 'longsleeve', 'shirt', 'shortpedding', 'shorts',
    'shortsleeve', 'skirt', 'slacks', 'sleeveless', 'sweater', 'trainingpants',
    'hoodzip', 'fleece', 'jumper'
]
NUM_CLASSES = len(class_names)
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

# 파라미터
IMG_SIZE = (416, 416)
SEED = 42

print(f"📂 Original data: {original_data_path}")
print(f"📂 New data: {new_data_path}")
print(f"💾 Preprocessed data will be saved to: {output_dir}")
print(f"📐 Input Image Size: {IMG_SIZE}")

# ✅ 4. 데이터 경로 수집
def create_df_from_dir(data_dir, dataset_name):
    """디렉토리를 스캔하여 파일 경로와 라벨의 DataFrame을 생성합니다."""
    filepaths, labels = [], []
    for class_name in class_names:
        class_folder = os.path.join(data_dir, class_name)
        if not os.path.exists(class_folder):
            print(f"⚠️ Warning: {class_folder} does not exist")
            continue
        for f in os.listdir(class_folder):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                filepaths.append(os.path.join(class_folder, f))
                labels.append(class_name)
    df = pd.DataFrame({'filepath': filepaths, 'label': labels})
    df['dataset'] = dataset_name
    return df

print("\n📂 Scanning image file paths...")
original_df = create_df_from_dir(original_data_path, 'original')
new_df = create_df_from_dir(new_data_path, 'new')
print(f"✅ Found {len(original_df)} original images and {len(new_df)} new images.")

# ✅ 5. 데이터셋 분할
def split_dataframe_stratified(df):
    """Stratified sampling으로 데이터를 Train/Val/Test로 분할합니다."""
    train_val_df, test_df = train_test_split(
        df, test_size=0.2, stratify=df['label'], random_state=SEED
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=0.125, stratify=train_val_df['label'], random_state=SEED
    )
    return train_df, val_df, test_df

print("\n📊 Splitting datasets with stratified sampling...")
original_train, original_val, original_test = split_dataframe_stratified(original_df)
new_train, new_val, new_test = split_dataframe_stratified(new_df)

train_df = pd.concat([original_train, new_train], ignore_index=True)
val_df = pd.concat([original_val, new_val], ignore_index=True)
test_df = pd.concat([original_test, new_test], ignore_index=True)

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\n✅ Data split complete:")
print(f"   - Training:   {len(train_df):,} images")
print(f"   - Validation: {len(val_df):,} images")
print(f"   - Test:       {len(test_df):,} images")

# ✅ 6. 이미지를 메모리에 로딩
def load_images_to_memory(df, split_name):
    """모든 이미지를 메모리에 로딩합니다."""
    print(f"\n⚡ Loading {split_name} images into memory...")
    images = []
    labels = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Loading {split_name}"):
        try:
            img = load_img(row['filepath'], target_size=IMG_SIZE)
            img_array = img_to_array(img)
            images.append(img_array)
            labels.append(class_to_idx[row['label']])
        except Exception as e:
            print(f"⚠️ Failed to load {row['filepath']}: {e}")
            continue

    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)

    print(f"✅ Loaded {len(images)} images into memory")
    print(f"   Memory usage: {images.nbytes / (1024**3):.2f} GB")

    return images, labels

# 데이터 로딩
X_train_raw, y_train_idx = load_images_to_memory(train_df, "Training")
X_val_raw, y_val_idx = load_images_to_memory(val_df, "Validation")
X_test_raw, y_test_idx = load_images_to_memory(test_df, "Test")

# 데이터프레임 삭제 (메모리 절약)
del train_df, val_df, test_df, original_df, new_df
gc.collect()

# ✅ 7. EfficientNet 전처리 적용
print("\n🔧 Applying EfficientNet preprocessing...")
X_train = preprocess_input(X_train_raw.copy())
X_val = preprocess_input(X_val_raw.copy())
X_test = preprocess_input(X_test_raw.copy())

print(f"✅ Preprocessing complete")
print(f"   Train range: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"   Val range: [{X_val.min():.2f}, {X_val.max():.2f}]")
print(f"   Test range: [{X_test.min():.2f}, {X_test.max():.2f}]")

# 원본 삭제 (메모리 절약)
del X_train_raw, X_val_raw, X_test_raw
gc.collect()

# ✅ 8. 라벨을 원-핫 인코딩
print("\n🏷️ Converting labels to one-hot encoding...")
y_train_onehot = to_categorical(y_train_idx, NUM_CLASSES)
y_val_onehot = to_categorical(y_val_idx, NUM_CLASSES)
y_test_onehot = to_categorical(y_test_idx, NUM_CLASSES)

print(f"✅ One-hot encoding complete")
print(f"   Train labels shape: {y_train_onehot.shape}")
print(f"   Val labels shape: {y_val_onehot.shape}")
print(f"   Test labels shape: {y_test_onehot.shape}")

# ✅ 9. Numpy 파일로 저장
print("\n💾 Saving preprocessed data to numpy files...")

# 전처리된 이미지 저장
np.save(os.path.join(output_dir, "X_train.npy"), X_train)
np.save(os.path.join(output_dir, "X_val.npy"), X_val)
np.save(os.path.join(output_dir, "X_test.npy"), X_test)

# 라벨 인덱스 저장 (정수형)
np.save(os.path.join(output_dir, "y_train_idx.npy"), y_train_idx)
np.save(os.path.join(output_dir, "y_val_idx.npy"), y_val_idx)
np.save(os.path.join(output_dir, "y_test_idx.npy"), y_test_idx)

# 원-핫 인코딩된 라벨 저장
np.save(os.path.join(output_dir, "y_train_onehot.npy"), y_train_onehot)
np.save(os.path.join(output_dir, "y_val_onehot.npy"), y_val_onehot)
np.save(os.path.join(output_dir, "y_test_onehot.npy"), y_test_onehot)

print("✅ All files saved successfully!")

# ✅ 10. 저장된 파일 정보 출력
print(f"\n{'='*70}")
print("📦 SAVED FILES SUMMARY")
print(f"{'='*70}")
print(f"\n📂 Save Location: {output_dir}\n")

file_info = [
    ("X_train.npy", X_train.shape, X_train.nbytes / (1024**3)),
    ("X_val.npy", X_val.shape, X_val.nbytes / (1024**3)),
    ("X_test.npy", X_test.shape, X_test.nbytes / (1024**3)),
    ("y_train_idx.npy", y_train_idx.shape, y_train_idx.nbytes / (1024**2)),
    ("y_val_idx.npy", y_val_idx.shape, y_val_idx.nbytes / (1024**2)),
    ("y_test_idx.npy", y_test_idx.shape, y_test_idx.nbytes / (1024**2)),
    ("y_train_onehot.npy", y_train_onehot.shape, y_train_onehot.nbytes / (1024**2)),
    ("y_val_onehot.npy", y_val_onehot.shape, y_val_onehot.nbytes / (1024**2)),
    ("y_test_onehot.npy", y_test_onehot.shape, y_test_onehot.nbytes / (1024**2)),
]

for filename, shape, size in file_info:
    if size < 1:  # MB 단위
        print(f"  ✓ {filename:25s} | Shape: {str(shape):20s} | Size: {size*1024:.2f} MB")
    else:  # GB 단위
        print(f"  ✓ {filename:25s} | Shape: {str(shape):20s} | Size: {size:.2f} GB")

total_size = sum([f[2] for f in file_info[:3]])  # 이미지 파일만
print(f"\n💾 Total Image Data Size: {total_size:.2f} GB")

print(f"\n{'='*70}")
print("✅ Data preprocessing and saving complete!")
print(f"{'='*70}")

# ✅ 11. 로딩 예제 코드 출력
print("\n📖 HOW TO LOAD THESE FILES:\n")
print("```python")
print("import numpy as np")
print("")
print("# 전처리된 이미지 로드")
print(f"X_train = np.load('{output_dir}/X_train.npy')")
print(f"X_val = np.load('{output_dir}/X_val.npy')")
print(f"X_test = np.load('{output_dir}/X_test.npy')")
print("")
print("# 라벨 로드 (정수형)")
print(f"y_train_idx = np.load('{output_dir}/y_train_idx.npy')")
print(f"y_val_idx = np.load('{output_dir}/y_val_idx.npy')")
print(f"y_test_idx = np.load('{output_dir}/y_test_idx.npy')")
print("")
print("# 라벨 로드 (원-핫 인코딩)")
print(f"y_train = np.load('{output_dir}/y_train_onehot.npy')")
print(f"y_val = np.load('{output_dir}/y_val_onehot.npy')")
print(f"y_test = np.load('{output_dir}/y_test_onehot.npy')")
print("")
print("print(f'Train: {X_train.shape}, {y_train.shape}')")
print("print(f'Val: {X_val.shape}, {y_val.shape}')")
print("print(f'Test: {X_test.shape}, {y_test.shape}')")
print("```")

Setting up environment...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment setup complete.

Importing libraries...
✅ Libraries imported successfully.

Configuring paths and parameters...
📂 Original data: /content/drive/MyDrive/윤서/윤서/dataset3/images
📂 New data: /content/drive/MyDrive/윤서/윤서/retraining_data
💾 Preprocessed data will be saved to: /content/drive/MyDrive/윤서/윤서/preprocessed_data
📐 Input Image Size: (416, 416)

📂 Scanning image file paths...
✅ Found 6888 original images and 1077 new images.

📊 Splitting datasets with stratified sampling...

✅ Data split complete:
   - Training:   5,574 images
   - Validation: 797 images
   - Test:       1,594 images

⚡ Loading Training images into memory...


Loading Training: 100%|██████████| 5574/5574 [41:24<00:00,  2.24it/s]


✅ Loaded 5574 images into memory
   Memory usage: 10.78 GB

⚡ Loading Validation images into memory...


Loading Validation: 100%|██████████| 797/797 [06:43<00:00,  1.97it/s]


✅ Loaded 797 images into memory
   Memory usage: 1.54 GB

⚡ Loading Test images into memory...


Loading Test: 100%|██████████| 1594/1594 [14:43<00:00,  1.80it/s]


✅ Loaded 1594 images into memory
   Memory usage: 3.08 GB

🔧 Applying EfficientNet preprocessing...
✅ Preprocessing complete
   Train range: [0.00, 255.00]
   Val range: [0.00, 255.00]
   Test range: [0.00, 255.00]

🏷️ Converting labels to one-hot encoding...
✅ One-hot encoding complete
   Train labels shape: (5574, 21)
   Val labels shape: (797, 21)
   Test labels shape: (1594, 21)

💾 Saving preprocessed data to numpy files...
✅ All files saved successfully!

📦 SAVED FILES SUMMARY

📂 Save Location: /content/drive/MyDrive/윤서/윤서/preprocessed_data

  ✓ X_train.npy               | Shape: (5574, 416, 416, 3)  | Size: 10.78 GB
  ✓ X_val.npy                 | Shape: (797, 416, 416, 3)   | Size: 1.54 GB
  ✓ X_test.npy                | Shape: (1594, 416, 416, 3)  | Size: 3.08 GB
  ✓ y_train_idx.npy           | Shape: (5574,)              | Size: 21.77 MB
  ✓ y_val_idx.npy             | Shape: (797,)               | Size: 3.11 MB
  ✓ y_test_idx.npy            | Shape: (1594,)              | Siz

#분류모델 학습

In [ ]:
#분류모델 학습

In [ ]:
# -*- coding: utf-8 -*-
"""
EfficientNetB3 Training Script - From Preprocessed .NPY Files

⚡ 전처리된 .npy 파일을 메모리에 로드하여 빠른 학습
"""

# ✅ 1. Google Drive 마운트
print("Setting up environment...")
from google.colab import drive
import os
drive.mount('/content/drive')
print("✅ Environment setup complete.")


# ✅ 2. 필수 라이브러리 임포트
print("\nImporting libraries...")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import pandas as pd
# import shutil  # 심볼릭 링크를 사용하지 않으므로 제거

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator # 증강을 위해 유지
# from sklearn.model_selection import train_test_split  # .npy에 이미 분할됨
from sklearn.metrics import confusion_matrix, classification_report
# ⭐ EfficientNetB3 임포트
from tensorflow.keras.applications import EfficientNetB3
# from tensorflow.keras.applications.efficientnet import preprocess_input  # .npy에 이미 적용됨
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

Image.MAX_IMAGE_PIXELS = None
print("✅ Libraries imported successfully.")


# ✅ 3. 경로 및 파라미터 설정
print("\nConfiguring paths and parameters...")
# --- 사용자 설정 경로 ---
# ⭐ .npy 파일이 저장된 경로
numpy_data_path = "/content/drive/MyDrive/윤서/윤서/preprocessed_data"
# ⭐ 결과 저장 폴더 (이전 결과와 겹치지 않도록 이름 변경)
output_dir = "/content/drive/MyDrive/윤서/윤서/training_output_efficientnetb3_from_npy"
# --- --- --- --- ---

# 원본 이미지 경로 및 임시 폴더 경로 제거
# original_data_path = "..."
# new_data_path = "..."
# structured_dataset_path = "..."

final_model_path = os.path.join(output_dir, "efficientnetb3_final_model.keras")
os.makedirs(output_dir, exist_ok=True)

class_names = [
    'blazer', 'cardigan', 'coat', 'cotton pants', 'denim', 'dress', 'hood',
    'longpedding', 'longsleeve', 'shirt', 'shortpedding', 'shorts',
    'shortsleeve', 'skirt', 'slacks', 'sleeveless', 'sweater', 'trainingpants',
    'hoodzip', 'fleece', 'jumper'
]
NUM_CLASSES = len(class_names)

# 학습 파라미터
IMG_SIZE = (416, 416) # .npy 생성 시 사용한 크기와 동일해야 함
BATCH_SIZE = 32 # 메모리 로딩이 빠르므로 배치 크기 증가 (16 -> 32)
SEED = 42
EPOCHS_STAGE1 = 25
EPOCHS_STAGE2 = 45 # 총 45 에포크 (1단계 25 + 2단계 20)
LR_STAGE1 = 0.0001
LR_STAGE2 = 0.00002

print(f"💾 Loading preprocessed data from: {numpy_data_path}")
print(f"💾 Model and reports will be saved to: {output_dir}")
print(f"📐 Input Image Size: {IMG_SIZE}")


# ✅ 4. 데이터 로딩 (Numpy 파일에서 직접 로드)
# 기존 4, 5, 6 단계 (경로 수집, 분할, 심볼릭 링크 생성)를 대체
print(f"\n⚡ Loading preprocessed data from '{numpy_data_path}'...")

try:
    X_train = np.load(os.path.join(numpy_data_path, 'X_train.npy'))
    y_train = np.load(os.path.join(numpy_data_path, 'y_train_onehot.npy'))

    X_val = np.load(os.path.join(numpy_data_path, 'X_val.npy'))
    y_val = np.load(os.path.join(numpy_data_path, 'y_val_onehot.npy'))

    X_test = np.load(os.path.join(numpy_data_path, 'X_test.npy'))
    y_test = np.load(os.path.join(numpy_data_path, 'y_test_onehot.npy'))

    # Classification Report를 위한 정수형 라벨
    y_test_idx = np.load(os.path.join(numpy_data_path, 'y_test_idx.npy'))

    print("✅ Data loaded successfully:")
    print(f"   - X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"   - X_val:   {X_val.shape}, y_val: {y_val.shape}")
    print(f"   - X_test:  {X_test.shape}, y_test: {y_test.shape}")
    print(f"   - y_test_idx: {y_test_idx.shape}")

    # .npy 로딩 후 메모리 정리
    gc.collect()

except FileNotFoundError as e:
    print(f"❌ ERROR: Could not find .npy file. Make sure the path is correct.")
    print(f"Error details: {e}")
    # 스크립트 중단 (혹은 적절한 오류 처리)
    raise e


# ✅ 5. 데이터 증강 제너레이터 생성
# (flow_from_directory 대체)
print("\n🎨 Creating data augmentation generator...")
train_datagen = ImageDataGenerator(
    rotation_range=30, zoom_range=0.25, width_shift_range=0.15,
    height_shift_range=0.15, horizontal_flip=True, brightness_range=[0.7, 1.3],
    shear_range=0.15, fill_mode='nearest'
    # ⭐ [중요] preprocessing_function=preprocess_input 제거
    # .npy 파일은 이미 전처리가 완료된 상태임
)

# 검증/테스트 데이터는 증강이 필요 없으며, 전처리도 이미 완료됨.
# 따라서 별도 제너레이터가 필요 없음. (fit/evaluate에 튜플로 바로 전달)

print("✅ Data augmentation generator is ready.")


# ✅ 6. 모델 구성 (EfficientNetB3)
# (기존 8번 항목)
def build_efficientnetb3_model(num_classes):
    print("\n🏗️ Building model with EfficientNetB3 base...")
    base_model = EfficientNetB3(
        include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3)
    )
    inputs = layers.Input(shape=(*IMG_SIZE, 3), name='input_layer')
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs, name="EfficientNetB3_Classifier")

model = build_efficientnetb3_model(NUM_CLASSES)
model.summary()


# ✅ 7. 모델 학습
# (기존 9번 항목)
print("\n🚀 Starting 2-stage model training...")
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

# --- 1단계: 헤드 학습 ---
print("\n--- Stage 1: Training Head (Frozen Backbone) ---")
model.layers[1].trainable = False # Freeze base_model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR_STAGE1), loss=loss_fn, metrics=['accuracy'])

history_stage1 = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS_STAGE1,
    validation_data=(X_val, y_val), # 튜플로 직접 전달
    callbacks=[
        EarlyStopping(patience=8, monitor='val_accuracy', restore_best_weights=True),
        ReduceLROnPlateau(patience=4, monitor='val_loss')
    ]
)

# --- 2단계: 미세 조정 ---
print("\n--- Stage 2: Fine-tuning (Unfrozen Backbone) ---")
model.layers[1].trainable = True # Unfreeze base_model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR_STAGE2), loss=loss_fn, metrics=['accuracy'])

history_stage2 = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS_STAGE2,
    validation_data=(X_val, y_val), # 튜플로 직접 전달
    initial_epoch=history_stage1.epoch[-1] + 1,
    callbacks=[
        EarlyStopping(patience=12, monitor='val_accuracy', restore_best_weights=True),
        ReduceLROnPlateau(patience=6, monitor='val_loss'),
        ModelCheckpoint(final_model_path, save_best_only=True, monitor='val_accuracy', verbose=1)
    ]
)
print("\n✅ Training complete!")


# ✅ 8. 최종 모델 평가
# (기존 10번 항목 수정 - TTA 제거)
print(f"\n📂 Reloading best weights from {final_model_path}...")
model.load_weights(final_model_path)

print("\n📊 Evaluating final model performance on test set...")
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"  • Standard Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# TTA 코드 제거 - .npy가 이미 전처리되어 TTA 적용이 부적절함
# 대신 표준 예측 수행
print("\n📊 Generating predictions for classification report...")
y_pred_probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test_idx # .npy에서 로드한 정수형 라벨 사용

# (x_test_tta, predict_with_tta 함수 등 TTA 관련 코드 모두 삭제)


# ✅ 9. 리포트 및 시각화
# (기존 11번 항목 수정 - TTA 제거)
print("\n📈 Final Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

print("\n📊 Generating Confusion Matrix...")
plt.figure(figsize=(22, 20))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
# 제목에서 TTA 제거, test_accuracy 사용
plt.title(f"Final Confusion Matrix (Accuracy: {test_accuracy:.4f})", fontsize=18)
plt.xlabel('Predicted Label', fontsize=14); plt.ylabel('True Label', fontsize=14)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
# 파일명에서 _tta 제거
plt.savefig(os.path.join(output_dir, "confusion_matrix_final_effnetb3_from_npy.png"), dpi=300)
plt.show()

# 메모리 확보
del X_train, y_train, X_val, y_val, X_test, y_test, y_test_idx
gc.collect()


# ✅ 10. 학습 과정 시각화 및 최종 리포트
# (기존 12번 항목)
print("\n📊 Visualizing training history...")
combined_history = {}
for key in history_stage1.history.keys():
    if key in history_stage2.history:
        combined_history[key] = history_stage1.history[key] + history_stage2.history[key]

stage1_epochs_run = len(history_stage1.history['loss'])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Model Training History (EfficientNetB3 from .NPY)', fontsize=18) # 제목 변경

ax1.plot(combined_history['accuracy'], label='Train Accuracy')
ax1.plot(combined_history['val_accuracy'], label='Validation Accuracy')
ax1.axvline(x=stage1_epochs_run - 1, color='r', linestyle='--', label='Start Fine-tuning')
ax1.set_title('Model Accuracy'); ax1.set_ylabel('Accuracy'); ax1.set_xlabel('Epoch')
ax1.legend(loc='lower right'); ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(combined_history['loss'], label='Train Loss')
ax2.plot(combined_history['val_loss'], label='Validation Loss')
ax2.axvline(x=stage1_epochs_run - 1, color='r', linestyle='--', label='Start Fine-tuning')
ax2.set_title('Model Loss'); ax2.set_ylabel('Loss'); ax2.set_xlabel('Epoch')
ax2.legend(loc='upper right'); ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(output_dir, "training_history_efficientnetb3_from_npy.png"), dpi=300) # 파일 이름 변경
plt.show()

print(f"\n{'='*70}")
print(f"🎉 FINAL TRAINING REPORT (EfficientNetB3 from .NPY)") # 제목 변경
print(f"{'='*70}")
print("\n📊 Dataset Summary (from .NPY):")
# len(train_df) -> 로드된 numpy 배열의 길이로 변경
print(f"  • Training Set:   {len(history_stage1.model.validation_data[0]):,}") # X_val 길이
val_count = len(history_stage1.model.validation_data[0])
test_count = len(y_pred)
train_count = val_count * (0.7 / 0.1) # 70:10 비율로 역산 (대략)
print(f"  • Training Set:   {int(train_count):,} (Approx.)")
print(f"  • Validation Set: {val_count:,}")
print(f"  • Test Set:       {test_count:,}")

print("\n📈 Performance Metrics:")
print(f"  • Standard Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
# TTA 라인 제거
best_val_acc_s1 = max(history_stage1.history.get('val_accuracy', [0]))
best_val_acc_s2 = max(history_stage2.history.get('val_accuracy', [0]))
print(f"  • Best Validation Accuracy (Stage 1): {best_val_acc_s1:.4f}")
print(f"  • Best Validation Accuracy (Stage 2): {best_val_acc_s2:.4f}")

print(f"\n💾 Saved Artifacts:")
print(f"  • Final Model: {final_model_path}")
print(f"  • Confusion Matrix: {os.path.join(output_dir, 'confusion_matrix_final_effnetb3_from_npy.png')}") # 파일 이름 변경
print(f"  • Training History Plot: {os.path.join(output_dir, 'training_history_efficientnetb3_from_npy.png')}") # 파일 이름 변경
print(f"\n{'='*70}")


Setting up environment...
